# Relation Extraction

Ensure that FastAPI is up and running. [instructions here](../fast_api/readme.md)

# Table of contents

1. Data prep
2. Standard
        2.1. Train Model
        2.2. Predict Labels
        2.3. Evaluate Model
3. NeXT
        3.1. Soft Match
            3.1.1. Train Model
            3.1.2. Predict Labels
            3.1.3. Evaluate Model
        3.2. Strict Match
            3.2.1. Train Model
            3.2.2. Predict Labels
            3.2.3. Evaluate Model

In [1]:
# imports
import requests
import json

In [2]:
FAST_API_URL = "http://localhost:9000"

 ---
 # 1. Data prep

We use a subset of `SemEval 2010 Task 8` dataset in this demo.

---
#### Prepare data for Standard training

In [3]:
# load labelled data without explanations for Standard training
re_standard_labeled_data = list(map(
    lambda x: {'text': x['text'], 'label': x['label']}, 
    json.load(open('data/semeval_labeled.json'))
))
re_standard_labeled_data[0]

{'text': 'The SUBJ-O is being discussed on message OBJ-O .',
 'label': 'Message-Topic(e2,e1)'}

In [4]:
re_standard_dev_data = json.load(open('data/semeval_dev.json'))
re_standard_dev_data[0]

{'text': 'Apartment residents in Delaware were awarded $ 1 million for health problems due to mold SUBJ-O from unrepaired OBJ-O .',
 'label': 'Cause-Effect(e2,e1)'}

---
#### Prepare data for NeXT training

In [5]:
# load unlabelled data for NeXT training
re_next_unlabeled_data = json.load(open('data/semeval_unlabeled.json'))
re_next_unlabeled_data[0]

'Throughout his writing career , all of his SUBJ-O have concerned fictional secret OBJ-O .'

In [6]:
# load labelled data with explanations for NeXT training
re_next_explanation_triples = json.load(open('data/semeval_labeled.json'))
re_next_explanation_triples[0]

{'label': 'Message-Topic(e2,e1)',
 'explanation': 'The phrase "being discussed" occurs between SUBJ and OBJ and there are no more than three words between SUBJ and OBJ and SUBJ precedes OBJ',
 'text': 'The SUBJ-O is being discussed on message OBJ-O .'}

---
#### Prepare data for evaluation

In [7]:
# load evaluation samples
re_eval_data = json.load(open('data/semeval_test.json'))
# [{text: str, label: str}]
re_eval_data = list(map(lambda doc: [doc['text'], doc['label']], re_eval_data))
# [Sentence, Label] pairs
re_eval_data[0]

['After the family leaves , remove the towel from under the SUBJ-O of the deceased OBJ-O .',
 'Component-Whole(e1,e2)']

---
#### Prepare data for prediction

In [8]:
re_predict_data = list(map(lambda x: x[0], re_eval_data))[:10]
re_predict_data[0]

'After the family leaves , remove the towel from under the SUBJ-O of the deceased OBJ-O .'

---
#### Define label space

In [9]:
# dictionary of (key: String, value: Int), key being the label and value being unique integer ID
re_label_space = {
  "Product-Producer(e1,e2)": 0,
  "Component-Whole(e2,e1)": 1,
  "Product-Producer(e2,e1)": 2,
  "Instrument-Agency(e2,e1)": 3,
  "Message-Topic(e1,e2)": 4,
  "Entity-Origin(e2,e1)": 5,
  "Entity-Origin(e1,e2)": 6,
  "Entity-Destination(e1,e2)": 7,
  "Cause-Effect(e1,e2)": 8,
  "Content-Container(e1,e2)": 9,
  "Member-Collection(e2,e1)": 10,
  "Instrument-Agency(e1,e2)": 11,
  "Message-Topic(e2,e1)": 12,
  "Content-Container(e2,e1)": 13,
  "no_relation": 14,
  "Member-Collection(e1,e2)": 15,
  "Entity-Destination(e2,e1)": 16,
  "Component-Whole(e1,e2)": 17,
  "Cause-Effect(e2,e1)": 18
}

---
#### Define NER label space

In [10]:
re_ner_label_space = json.load(open('data/semeval_ner_space.json'))
re_ner_label_space

['ORGANIZATION',
 'PERSON',
 'MONEY',
 'NATIONALITY',
 'SET',
 'COUNTRY',
 'NUMBER',
 'MISC',
 'CAUSE_OF_DEATH',
 'STATE_OR_PROVINCE',
 'DATE',
 'IDEOLOGY',
 'ORDINAL',
 'RELIGION',
 'LOCATION',
 'TIME',
 'DURATION',
 'CRIMINAL_CHARGE',
 'TITLE',
 'O']

---
---
# 2. Standard

Check [this](../fast_api/json_schema.py#L516) json schema for a list of all parameters.

Check [this](../model_training/internal_api/defaults.py) for default values.

## 2.1. Train model

### Define training parameters

In [11]:
re_standard_train_params = {
    # a string name representing the model name
    "experiment_name": "semeval_re_standard",
    # a string name representing the dataset name
    "dataset_name": "semeval_re_standard",
    # task type - "re" for Relation Extraction    
    "task": "re",
    # training batch size
    "match_batch_size": 50,
    # learning rate
    "learning_rate": 0.1,
    # number of epochs
    "epochs": 5,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100,
    # random seed
    "random_state": 7698,
    # when "True" data has to be passed, 
    # "False" when re-training or the data was processed earlier and can be retrieved
    "build_data": True,
    # label representing the relation "no_relation"
    "none_label_key": "no_relation",
}

### Run model training

In [13]:
# depending on input size, and computing environment this might take time.
# please check FAST API logs for updates
response = requests.post(
    FAST_API_URL + '/training/standard/api/',
    json={
        'params': re_standard_train_params,
        'label_space': re_label_space,
        'labeled_data': re_standard_labeled_data,
        'dev_data': re_standard_dev_data,
        'ner_label_space': re_ner_label_space
    }
)
# JSON with "save_path" key is returned when successful
response.text

'{"save_path":"/home/LEAN-LIFE/model_api/fast_api/../model_training/internal_api/../next_framework/data/saved_models/Clf_semeval_re_standard.p"}'

---
## 2.2. Predict labels

### Define prediction parameters

In [14]:
re_standard_predict_params = {
    # a string name representing the model name
    "experiment_name": "semeval_re_standard",
    # a string name representing the dataset name
    "dataset_name": "semeval_re_standard",
    # task type - "re" for Relation Extraction
    "task": "re",
    # evaluation batch size
    "eval_batch_size": 50,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100,
    # label representing the relation "no_relation"
    "none_label_key": "no_relation",
}

### Fetch predictions

In [15]:
response = requests.post(
    FAST_API_URL + '/training/standard/predict/',
    json={
        'params': re_standard_predict_params,
        'label_space': re_label_space,
        'prediction_data': re_predict_data,
        'ner_label_space': re_ner_label_space
    }
)
# returns JSON object with keys
# "class_probs" - list of list representing predictions for each label
# "class_preds" - list of integers representing the label that was predicted
response.text

'{"class_probs":[[0.0518844798207283,0.0523955412209034,0.03724173456430435,0.06492412835359573,0.06046446040272713,0.03642776608467102,0.10146567225456238,0.06655306369066238,0.07503866404294968,0.03900397941470146,0.02870575524866581,0.05798938870429993,0.038751497864723206,0.031560756266117096,0.024733280763030052,0.032614462077617645,0.02651955559849739,0.03288751468062401,0.1408383548259735],[0.05314013361930847,0.05230709910392761,0.0367225743830204,0.06612556427717209,0.06000146642327309,0.035538408905267715,0.10185324400663376,0.06383846700191498,0.07709857076406479,0.03838886693120003,0.028310289606451988,0.05783988907933235,0.0385381281375885,0.031566914170980453,0.023658009245991707,0.03236681967973709,0.025676395744085312,0.032598674297332764,0.14443042874336243],[0.05300821736454964,0.052461378276348114,0.036406613886356354,0.06584180146455765,0.059390727430582047,0.035208601504564285,0.10300138592720032,0.06436558067798615,0.07709814608097076,0.03787129372358322,0.0277472

---
## 2.3. Evaluate model

### Define evaluation parameters

In [16]:
re_standard_eval_params = {
    # a string name representing the model name
    "experiment_name": "semeval_re_standard",
    # a string name representing the dataset name
    "dataset_name": "semeval_re_standard",
    # task type - "re" for Relation Extraction
    "task": "re",
    # evaluation batch size
    "eval_batch_size": 50,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100,
    # label representing the relation "no_relation"
    "none_label_key": "no_relation",
}

### Run model evaluation

In [17]:
response = requests.post(
    FAST_API_URL + '/training/standard/eval/',
    json={
        'params': re_standard_eval_params,
        'label_space': re_label_space,
        'eval_data': re_eval_data,
        'ner_label_space': re_ner_label_space
    }
)
# returns JSON object with keys
# "avg_loss" and "avg_eval_f1_score"
response.text

'{"avg_loss":3.07345,"avg_eval_f1_score":0.06451612903225805}'

---
---
# 3. NeXT

Check [this](../fast_api/json_schema.py#L293) json schema for a list of all parameters.

Check [this](../model_training/internal_api/defaults.py) for default values.

## 3.1. Soft Match

## 3.1.1. Train model

### Define training parameters

In [28]:
re_next_soft_match_params = {
    # a string name representing the model name
    "experiment_name": "semeval_re_nle_soft_match",
    # a string name representing the dataset name
    "dataset_name": "semeval_re_nle_soft_match",
    # integer representing the unlabelled dataset size
    "dataset_size": len(re_next_unlabeled_data),
    # task type - "re" for Relation Extraction
    "task": "re",
    # the total batch_size is "match_batch_size" + "unlabeled_batch_size"
    "match_batch_size": 50,
    "unlabeled_batch_size": 100,
    # learning rate
    "learning_rate": 0.1,
    # number of epochs
    "epochs": 5,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d", # usual default: "glove.840B.300d"
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # soft_match loss weightage
    "gamma": 0.5,
    # number of hidden dimensions
    "hidden_dim": 100,
    # random seed
    "random_state": 7698,
    # FIND module number of hidden dimensions
    "pre_train_hidden_dim": 300,
    # FIND module training_size
    "pre_train_training_size": 50000,
    
    # enable soft match
    "soft_match": True,
    
    # "find" to train only FIND module, 
    # "clf" to train only the classifier,
    # "both" to train FIND module + classifier
    "stage": "both",
    
    # for the FIND module,
    # when "True" data has to be passed, 
    # "False" when re-training or the data was processed earlier and can be retrieved
    "pre_train_build_data": True,
    
    # when "True" data has to be passed, 
    # "False" when re-training or the data was processed earlier and can be retrieved
    "build_data": True,
    
    "pre_train_epochs": 5,
}

### Run model training

In [29]:
# depending on input size, and computing environment this might take time.
# please check FAST API logs for updates
response = requests.post(
    FAST_API_URL + '/training/next/api/',
    json={
        'params': re_next_soft_match_params,
        'label_space': re_label_space,
        'explanation_triples': re_next_explanation_triples,
        'unlabeled_text': re_next_unlabeled_data,
        'ner_label_space': re_ner_label_space
    }
)
# JSON with "save_path" key is returned when successful
response.text

'Internal Server Error'

---
## 3.1.2. Predict labels

### Define prediction parameters

In [ ]:
re_next_soft_predict_params = {
    # a string name representing the model name
    "experiment_name": "semeval_re_nle_soft_match",
    # a string name representing the dataset name
    "dataset_name": "semeval_re_nle_soft_match",
    # integer representing the unlabelled dataset size
    "train_dataset_size": len(re_next_unlabeled_data),
    # task type - "re" for Relation Extraction
    "task": "re",
    # evaluation batch size
    "eval_batch_size": 50,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d", # usual default: "glove.840B.300d"
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100,
    # FIND module training_size
    "pre_train_training_size": 50000,
    # label representing the relation "no_relation"
    "none_label_key": "no_relation",
}

### Fetch predictions

In [ ]:
response = requests.post(
    FAST_API_URL + '/training/next/predict/',
    json={
        'params': re_next_soft_predict_params,
        'label_space': re_label_space,
        'prediction_data': re_predict_data,
        'ner_label_space': re_ner_label_space
    }
)
# returns JSON object with keys
# "class_probs" - list of list representing predictions for each label
# "class_preds" - list of integers representing the label that was predicted
response.text

---
## 3.1.3. Evaluate model

### Define evaluation parameters

In [ ]:
re_next_soft_eval_params = {
    # a string name representing the model name
    "experiment_name": "semeval_re_nle_soft_match",
    # a string name representing the dataset name
    "dataset_name": "semeval_re_nle_soft_match",
    # integer representing the unlabelled dataset size
    "train_dataset_size": len(re_next_unlabeled_data),
    # task type - "re" for Relation Extraction
    "task": "re",
    # evaluation batch size
    "eval_batch_size": 50,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d", # usual default: "glove.840B.300d"
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100,
    # FIND module training_size
    "pre_train_training_size": 50000,
    # label representing the relation "no_relation"
    "none_label_key": "no_relation",
}

### Run model evaluation

In [ ]:
response = requests.post(
    FAST_API_URL + '/training/next/eval/',
    json={
        'params': re_next_soft_eval_params,
        'label_space': re_label_space,
        'eval_data': re_eval_data,
        'ner_label_space': re_ner_label_space
    }
)
# returns JSON object with keys
# "avg_loss", "avg_eval_ent_f1_score", "avg_eval_ent_f1_score" and "no_relation_thresholds"
response.text

---
---
## 3.2. Strict Match

## 3.2.1. Train model

### Define training parameters

In [20]:
# parameters are same as in "soft_match" except "soft_match" is set to "False" here
re_next_strict_match_params = {
    # a string name representing the model name
    "experiment_name": "semeval_re_nle_strict_match",
    # a string name representing the dataset name
    "dataset_name": "semeval_re_nle_strict_match",
    # integer representing the unlabelled dataset size
    "dataset_size": len(re_next_unlabeled_data),
    # task type - "re" for Relation Extraction
    "task": "re",
    # the total batch_size is "match_batch_size" + "unlabeled_batch_size"
    "match_batch_size": 50,
    "unlabeled_batch_size": 100,
    # learning rate
    "learning_rate": 0.1,
    # number of epochs
    "epochs": 5,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d", # usual default: "glove.840B.300d"
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # soft_match loss weightage
    "gamma": 0.5,
    # number of hidden dimensions
    "hidden_dim": 100,
    # random seed
    "random_state": 7698,
    # FIND module number of hidden dimensions
    "pre_train_hidden_dim": 300,
    # FIND module training_size
    "pre_train_training_size": 50000,
    
    # enable soft match
    "soft_match": False,
    
    # "find" to train only FIND module, 
    # "clf" to train only the classifier,
    # "both" to train FIND module + classifier
    "stage": "both",
    
    # for the FIND module,
    # when "True" data has to be passed, 
    # "False" when re-training or the data was processed earlier and can be retrieved
    "pre_train_build_data": True,
    
    # when "True" data has to be passed, 
    # "False" when re-training or the data was processed earlier and can be retrieved
    "build_data": True,
    
    # label representing the relation "no_relation"
    "none_label_key": "no_relation",
}

### Run model training

In [21]:
# depending on input size, and computing environment this might take time.
# please check FAST API logs for updates
response = requests.post(
    FAST_API_URL + '/training/next/api/',
    json={
        'params': re_next_strict_match_params,
        'label_space': re_label_space,
        'explanation_triples': re_next_explanation_triples,
        'unlabeled_text': re_next_unlabeled_data,
        'ner_label_space': re_ner_label_space
    }
)
# JSON with "save_path" key is returned when successful
response.text

'{"save_path":"/home/LEAN-LIFE/model_api/fast_api/../model_training/internal_api/../next_framework/data/saved_models/Clf_semeval_re_nle_strict_match.p"}'

---
## 3.2.2. Predict labels

### Define prediction parameters

In [22]:
re_next_strict_predict_params = {
    # a string name representing the model name
    "experiment_name": "semeval_re_nle_strict_match",
    # a string name representing the dataset name
    "dataset_name": "semeval_re_nle_strict_match",
    # task type - "re" for Relation Extraction
    "task": "re",
    # evaluation batch size
    "eval_batch_size": 50,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100,
    
    # label representing the relation "no_relation"
    "none_label_key": "no_relation",
}

### Fetch predictions

In [23]:
response = requests.post(
    FAST_API_URL + '/training/standard/predict/',
    json={
        'params': re_next_strict_predict_params,
        'label_space': re_label_space,
        'prediction_data': re_predict_data,
        'ner_label_space': re_ner_label_space
    }
)
# returns JSON object with keys
# "class_probs" - list of list representing predictions for each label
# "class_preds" - list of integers representing the label that was predicted
response.text

'{"class_probs":[[0.0397772453725338,0.05442820489406586,0.0517304353415966,0.03913586586713791,0.04108264669775963,0.043243568390607834,0.038859449326992035,0.05683298408985138,0.034561045467853546,0.044593632221221924,0.13594242930412292,0.04629042372107506,0.04008190333843231,0.03278108686208725,0.03713533654808998,0.03848375380039215,0.04011106491088867,0.10045164078474045,0.0844772681593895],[0.039910104125738144,0.0541856549680233,0.05175107717514038,0.03921910375356674,0.04012296348810196,0.04375739395618439,0.038221150636672974,0.055170051753520966,0.0345785953104496,0.04408350959420204,0.14000916481018066,0.0458124615252018,0.04093340411782265,0.03264252096414566,0.0369422622025013,0.03852967545390129,0.03988058492541313,0.09989157319068909,0.08435869961977005],[0.03995440527796745,0.05445996671915054,0.051691342145204544,0.03867869824171066,0.03979163616895676,0.043010834604501724,0.03819561004638672,0.05541967228055,0.03398051857948303,0.04362107440829277,0.14083977043628693

---
## 3.2.3. Evaluate model

### Define evaluation parameters

In [24]:
re_next_strict_eval_params = {
    # a string name representing the model name
    "experiment_name": "semeval_re_nle_strict_match",
    # a string name representing the dataset name
    "dataset_name": "semeval_re_nle_strict_match",
    # task type - "re" for Relation Extraction
    "task": "re",
    # evaluation batch size
    "eval_batch_size": 50,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100
}

### Run model evaluation

In [25]:
response = requests.post(
    FAST_API_URL + '/training/standard/eval/',
    json={
        'params': re_next_strict_eval_params,
        'label_space': re_label_space,
        'eval_data': re_eval_data,
        'ner_label_space': re_ner_label_space
    }
)
# returns JSON object with keys
# "avg_loss" and "avg_eval_f1_score"
response.text

'{"avg_loss":2.89852,"avg_eval_f1_score":0.10000000000000002}'